In [ ]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

: 

In [ ]:
# read excel sheet with a password  
def read_protected_excel(file_path, password, sheet_name):  
    """Read a sheet from a password-protected Excel file."""  
    with open(file_path, 'rb') as f:  
        office_file = msoffcrypto.OfficeFile(f)  
        office_file.load_key(password=password)  
        decrypted = io.BytesIO()  
        office_file.decrypt(decrypted)

    decrypted.seek(0)  
    df = pd.read_excel(decrypted, sheet_name=sheet_name, skiprows=5)

    return df


# set the file path for manpower files  
base_path = 'L:/2026 Pilar Plant Files/2026 Manpower Detail/'  
password = 'bamhorse2'  
# set the tab that is needed  
sheet_name = 'PPE LDR'  
# specify required filename keyword  
required_name = 'LENOX SALAERN'


# --- Generate all biweekly dates (not 15th or last day of month) ---  
import calendar

def get_biweekly_dates(start_date, end_date):  
    """Return all dates between start and end that are NOT the 15th or last day of the month."""  
    biweekly_dates = []  
    current = start_date  
    while current <= end_date:  
        last_day = calendar.monthrange(current.year, current.month)[1]  
        if current.day != 15 and current.day != last_day:  
            biweekly_dates.append(current)  
        current += pd.Timedelta(days=1)  
    return biweekly_dates


# define your date range (adjust as needed)  
start_date = pd.Timestamp('2026-01-01')  
end_date = pd.Timestamp('2026-12-31')

biweekly_dates = get_biweekly_dates(start_date, end_date)

# find all files matching each biweekly date  
all_biweekly_files = []  
for dt in biweekly_dates:  
    date_str = dt.strftime('%m-%d-%Y')  # adjust format to match your filenames  
    matched_files = glob.glob(os.path.join(base_path, f'*{date_str}*.xlsm'))  
    for f in matched_files:  
        # only include files that have the required name in the filename  
        if required_name.upper() in os.path.basename(f).upper():  
            all_biweekly_files.append((f, date_str))

print(f"\nBiweekly files found: {len(all_biweekly_files)}")  
for f, d in all_biweekly_files:  
    print(f"  {os.path.basename(f)} (date: {d})")


# combine into one list with pay type labels  
files = [(f, 'Biweekly', d) for f, d in all_biweekly_files]


all_ppe_ldr = []

# for loop!  
for file_path, pay_type, date in files:  
    # extract site name from filename (everything before the date)  
    filename = os.path.basename(file_path)  
    site = filename.split(' ')[0].strip()

    print(f"\nReading {pay_type} | Site: {site} | Date: {date}")

    try:  
        df = read_protected_excel(file_path, password, sheet_name)  
        # adding extra columns  
        df['Pay_Type'] = pay_type  
        df['Report_Date'] = pd.to_datetime(date)  
        df['BU'] = site

        # appending data to dataframe  
        all_ppe_ldr.append(df)

    except Exception as e:  
        print(f"  ERROR: {e}")


# combines lists of dataframes into one dataframe  
ppe_ldr_combined = pd.concat(all_ppe_ldr, ignore_index=True)

# renames columns to preferred format  
ppe_ldr_combined.columns = (  
    ppe_ldr_combined.columns.str.strip()  
    .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)  
    .str.replace(r'_+', '_', regex=True)  
    .str.strip('_')  
    .str.lower()  
)

print(f"\n--- Summary ---")  
print(f"PPE LDR:  {ppe_ldr_combined.shape}")  
print(f"Sites found: {ppe_ldr_combined['bu'].unique()}")